# How to use our models

In [1]:
import sys
import os

project_path = "/home/sagemaker-user/gbm_hackathon"
if project_path not in sys.path:
    sys.path.append(project_path)
    print(sys.path)

['/home/sagemaker-user/.conda/envs/gbmhackathon/lib/python310.zip', '/home/sagemaker-user/.conda/envs/gbmhackathon/lib/python3.10', '/home/sagemaker-user/.conda/envs/gbmhackathon/lib/python3.10/lib-dynload', '', '/home/sagemaker-user/.conda/envs/gbmhackathon/lib/python3.10/site-packages', '/home/sagemaker-user/gbm_hackathon']


In [9]:
%load_ext autoreload
%autoreload 2
    
from gbmhackathon.models.mme import *
from gbmhackathon.s3_loader import load_s3

import os
from copy import deepcopy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F 
from torch.utils.data import DataLoader
from torch.optim import Adam

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 3 types of network at disposal
You can chose from 3 types of **configurable** architectures:
- ### `MLP` -> MultiLayerPerceptron with parameters:
```python
class MLP(nn.Module):
    """Multi Layer Perceptron with dropout and normalizaiton layers that can be instantiated dynamically"""
    def __init__(
            self,
            layers: List[int],
            dropout: List[float] | float,
            act_fn: List[Callable | None] | Callable | None,
            norm_layer: List[Callable | None] | Callable | None,
            enable_residuals: bool = True,
        ):
        """
        Parameters:
        -----------
        layers : List[int]
            A sequence of layer sizes, including input and output dimensions.
            e.g. [in_dim, hidden1, hidden2, ..., out_dim]

        dropout : float or List[float]
            Dropout probability (0–1) applied after each hidden Linear layer.
            If a single float is given, the same rate is used everywhere;
            if a list, its length should match the number of hidden layers.

        act_fn : Callable or List[Callable or None]
            Activation function(s) to insert after each hidden layer.
            Can be a single callable (e.g. nn.ReLU) or a list of the same length
            as the hidden layers, with None to skip activation at specific layers.

        norm_layer : Callable or List[Callable or None]
            Normalization layer(s) to apply after activation.
            Accepts a layer constructor (e.g. nn.LayerNorm) or a matching list
            (with None entries to skip normalization).

        enable_residuals : bool (default: True)
            If True, automatically add skip‑connections between any two Linear layers
            that share the same dimensionality to help gradient flow.
        """
```

- ### `AttentionNetwork` -> An attention based encoder with GatedGELU, Pre-Normalization and DropPath (randomly drops entire residual branches during training with a probability that increases with network depth for better regularization) with parameters:
```python
class AttentionNetwork(nn.Module):
    def __init__(
        self,
        *,
        dim: int,
        depth: int,
        num_heads: int,
        mlp_ratio: float = 4.0,
        qkv_bias: bool = True,
        attn_dropout: float = 0.0,
        proj_dropout: float = 0.0,
        mlp_dropout: float = 0.0,
        drop_path_rate: float = 0.1,
    ):
        """
        Parameters:
        -----------
        dim : int
            Dimensionality of the input and output features.

        depth : int
            Number of sequential Transformer-style attention blocks.

        num_heads : int
            Number of attention heads in each MultiHeadAttention block.

        mlp_ratio : float (default: 4.0)
            Expansion ratio for the hidden layer size in the MLP block relative to the input dimension.
            For example, if `dim=64` and `mlp_ratio=4.0`, the hidden layer in the MLP will have 64 * 4 = 256 units.

        qkv_bias : bool (default: True)
            If True, enables learnable bias for query, key, and value projections.

        attn_dropout : float (default: 0.0)
            Dropout applied to attention weights.

        proj_dropout : float (default: 0.0)
            Dropout applied after the output projection of the attention block.

        mlp_dropout : float (default: 0.0)
            Dropout applied after the activation in the MLP block.

        drop_path_rate : float (default: 0.1)
            Drop path probability used for stochastic depth regularization.
        """
```
- ### `GraphEncoder` -> A GraphEncoder with parameters:
```python
class GraphEncoder(nn.Module):
    def __init__(self,
                in_channels : int,
                hidden_channels : int,
                out_channels : int,
                dropout : float,
                mean_pool : bool = False, 
                activation_post_gat : Callable = F.relu,
                att_agg_activation : Callable = nn.ReLU,
                heads : int = 1,
                ):
        """
        Parameters:
        -----------
        in_channels : int
            Number of input features per node.

        hidden_channels : int
            Number of hidden features in the GAT layer (per head).

        out_channels : int
            Number of output features for the final representation.

        dropout : float
            Dropout probability applied after the GAT activation.

        mean_pool : bool (default: False)
            If True, use global mean pooling; otherwise, use attention-based pooling.

        activation_post_gat : Callable (default: F.relu)
            Activation function applied after the GAT layer.

        att_agg_activation : Callable (default: nn.ReLU)
            Activation function used inside the attention gate MLP for pooling.

        heads : int (default: 1)
            Number of attention heads in the GAT layer.
        """
```

## They can be used by themselves see example below:
**Rule of thumb: Use config dictionaries to instantiate them.** 

In [13]:
out = 64

mlp_cfg = {
    "layers": [1024, 512, out],
    "dropout": 0.65,
    "act_fn": nn.ReLU,
    "norm_layer": nn.LayerNorm,
}

attn_cfg = {
    "dim": out,               # input & output feature size
    "depth": 4,               # number of transformer blocks
    "num_heads": 8,
    "mlp_ratio": 4.0,         # typical transformer MLP expansion
    "qkv_bias": True,
    "attn_dropout": 0.1,
    "proj_dropout": 0.1,
    "mlp_dropout": 0.1,
    "drop_path_rate": 0.2,
}

graph_cfg = {
    "in_channels": 64,            # input features per node
    "hidden_channels": 128,       # per-head hidden dim
    "out_channels": out,          # final output
    "dropout": 0.5,
    "mean_pool": False,
    "activation_post_gat": F.relu,
    "att_agg_activation": nn.ReLU,
    "heads": 4,
}

In [20]:
mlp = MLP(**mlp_cfg)
mlp

No potential residual connections found


MLP(
  (network_layers): ModuleList(
    (0): Linear(in_features=1024, out_features=512, bias=True)
    (1): Dropout(p=0.65, inplace=False)
    (2): ReLU()
    (3): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    (4): Linear(in_features=512, out_features=64, bias=True)
    (5): ReLU()
    (6): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
  )
)

In [17]:
attnet = AttentionNetwork(**attn_cfg)
attnet

AttentionNetwork(
  (blocks): ModuleList(
    (0-3): 4 x ModuleList(
      (0): AttentionBlock(
        (qkv): Linear(in_features=64, out_features=192, bias=True)
        (attn_drop): Dropout(p=0.1, inplace=False)
        (proj): Linear(in_features=64, out_features=64, bias=True)
        (proj_drop): Dropout(p=0.1, inplace=False)
        (norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (drop_path): DropPath()
      )
      (1): FeedForward(
        (norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (fc1): Linear(in_features=64, out_features=512, bias=True)
        (act): GEGLU()
        (fc2): Linear(in_features=256, out_features=64, bias=True)
        (drop): Dropout(p=0.1, inplace=False)
        (drop_path): DropPath()
      )
    )
  )
  (norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
)

In [19]:
graphnet = GraphEncoder(**graph_cfg)
graphnet

/home/sagemaker-user/.conda/envs/gbmhackathon/lib/python3.10/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'nn.glob.GlobalAttention' is deprecated, use 'nn.aggr.AttentionalAggregation' instead
  warnings.warn(out)


GraphEncoder(
  (gat): GATConv(64, 128, heads=4)
  (dropout): Dropout(p=0.5, inplace=False)
  (pooling): GlobalAttention(gate_nn=Sequential(
    (0): Linear(in_features=512, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=1, bias=True)
  ), nn=None)
  (fc): Linear(in_features=512, out_features=64, bias=True)
)

## ModalityEncoder module
This module is a wrapper around the three previous classes. It enables to seamlessly instantiate a modality encoder with the network and configuration of your choice.
Required fields are: 
- `"net_type"`: Either `mlp`, `attention` or `graph`, anything else will raise an error
- `"device"`: Can be a predifine device as in the example but can also be left to `None`
- `"net_config"`: The configuration dictionary of the network. Must be appropriate with respect to the type of Architecture specified by the "net_type" field. i.e if you pass a config for `MLP` when you specified `attention`, it will raise an error.

**!! PAY ATTENTION TO THE MODALITY INPUT SIZE SO YOUR CONFIG MATCHES !!**

When device is `None`, dynamic device fetching.

In [34]:
device = None
bulk_encoder_cfg = {"net_type": "mlp",
                    "device": device,
                    "net_config": mlp_cfg}
bulk_encoder = ModalityEncoder(bulk_encoder_cfg)
bulk_encoder

Using device: cpu
No potential residual connections found


ModalityEncoder(
  (net): MLP(
    (network_layers): ModuleList(
      (0): Linear(in_features=1024, out_features=512, bias=True)
      (1): Dropout(p=0.65, inplace=False)
      (2): ReLU()
      (3): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (4): Linear(in_features=512, out_features=64, bias=True)
      (5): ReLU()
      (6): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
    )
  )
)

In [35]:
device = "cuda" if torch.cuda.is_available() else "cpu"
bulk_encoder_cfg = {"net_type": "mlp",
                    "device": device,
                    "net_config": mlp_cfg}
bulk_encoder = ModalityEncoder(bulk_encoder_cfg)
bulk_encoder

Using device: cpu
No potential residual connections found


ModalityEncoder(
  (net): MLP(
    (network_layers): ModuleList(
      (0): Linear(in_features=1024, out_features=512, bias=True)
      (1): Dropout(p=0.65, inplace=False)
      (2): ReLU()
      (3): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (4): Linear(in_features=512, out_features=64, bias=True)
      (5): ReLU()
      (6): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
    )
  )
)

## MultiModalEncoder module
In practice, this the object we will be using the most. It creates a MultiModalEncoder with flexible configurations of each encoder.

In [36]:
bulk_cfg = {"net_type": "attention",
                    "device": device,
                    "net_config": attn_cfg}
hne_cfg = {"net_type": "mlp",
                    "device": device,
                    "net_config": mlp_cfg}
sc_cfg = {"net_type": "attention",
                    "device": device,
                    "net_config": attn_cfg}
wes_cfg = {"net_type": "attention",
                    "device": device,
                    "net_config": attn_cfg}
clinical_cfg = {"net_type": "mlp",
                    "device": device,
                    "net_config": mlp_cfg}
# spatial_cfg = {"net_type": "graph",
#                     "device": device,
#                     "net_config": graph_cfg}

In [37]:
mme_cfg = {"hne_cfg":hne_cfg, 
           "clinical_cfg":clinical_cfg, 
           "wes_cfg":wes_cfg, 
           #"spatial_cfg":spatial_cfg,
           "bulk_cfg":bulk_cfg,
          "sc_cfg":sc_cfg}

In [38]:
mme = MultiModalEncoder(**mme_cfg)
mme

Using device: cpu
No potential residual connections found
Using device: cpu
Using device: cpu
Using device: cpu
Using device: cpu
No potential residual connections found


MultiModalEncoder(
  (modality_net_map): ModuleDict(
    (hne): ModalityEncoder(
      (net): MLP(
        (network_layers): ModuleList(
          (0): Linear(in_features=1024, out_features=512, bias=True)
          (1): Dropout(p=0.65, inplace=False)
          (2): ReLU()
          (3): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (4): Linear(in_features=512, out_features=64, bias=True)
          (5): ReLU()
          (6): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        )
      )
    )
    (scRNA): ModalityEncoder(
      (net): AttentionNetwork(
        (blocks): ModuleList(
          (0-3): 4 x ModuleList(
            (0): AttentionBlock(
              (qkv): Linear(in_features=64, out_features=192, bias=True)
              (attn_drop): Dropout(p=0.1, inplace=False)
              (proj): Linear(in_features=64, out_features=64, bias=True)
              (proj_drop): Dropout(p=0.1, inplace=False)
              (norm): LayerNorm((64,), eps=1e-05, elementwi